In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
tabla_bronze    = f"{catalogo}.bronze.estaciones_raw"
tabla_silver     = f"{catalogo}.silver.estaciones"
tabla_cuarentena = f"{catalogo}.silver.estaciones_cuarentena"

In [0]:
# Límites aproximados de la CDMX para detectar coordenadas absurdas
LAT_MIN, LAT_MAX = 19.0, 19.7
LON_MIN, LON_MAX = -99.4, -98.9

In [0]:
bronze = spark.table(tabla_bronze )

In [0]:
def a_entero(columna):
    """Texto -> entero, o NULL si no tiene forma de número entero."""
    texto = F.trim(columna)
    return F.when(texto.rlike(r"^\d{1,9}$"), texto.cast("int"))


def a_decimal(columna):
    """Texto -> decimal, o NULL si no tiene forma de número (por ejemplo '19.4326' o '-99.1')."""
    texto = F.trim(columna)
    return F.when(texto.rlike(r"^[+-]?\d+(\.\d+)?$"), texto.cast("double"))

In [0]:
rangos = (
    bronze
    .filter(F.col("num_cicloe").contains("-"))
    .select(
        "_id", "num_cicloe", "calle_prin", "calle_secu",
        a_entero(F.split("num_cicloe", "-")[0]).alias("desde"),
        a_entero(F.split("num_cicloe", "-")[1]).alias("hasta"),
    )
    .withColumn("diferencia", F.col("hasta") - F.col("desde"))
)

In [0]:
print("Rangos encontrados:", rangos.count())
print("Rangos con diferencia distinta de 1 (revisar):")
display(rangos.filter((F.col("diferencia") != 1) | F.col("diferencia").isNull()))

Rangos encontrados: 12
Rangos con diferencia distinta de 1 (revisar):


_id,num_cicloe,calle_prin,calle_secu,desde,hasta,diferencia
77,264-275,Heroes Ferrocarrileros,Jesus Garcia,264,275,11


In [0]:
tipado = (
    bronze
    .withColumn("num_cicloe_original", F.trim("num_cicloe"))
    # "390-391" -> dos filas: "390" y "391". Un número normal ("423") sigue siendo una fila.
    .withColumn("num_cicloe", F.explode(F.split("num_cicloe_original", "-")))
    .select(
        a_entero("_id").alias("id_registro"),
        a_entero("num_cicloe").alias("num_cicloe"),
        "num_cicloe_original",
        # Identificador de la ubicación física: el primer número del rango (390-391 -> 390 para ambas filas)
        a_entero(F.split("num_cicloe_original", "-")[0]).alias("id_ubicacion"),
        F.trim("sistema").alias("sistema"),
        F.initcap(F.trim("calle_prin")).alias("calle_principal"),
        F.initcap(F.trim("calle_secu")).alias("calle_secundaria"),
        F.initcap(F.trim("colonia")).alias("colonia"),
        F.initcap(F.trim("alcaldia")).alias("alcaldia"),
        a_decimal("latitud").alias("latitud"),
        a_decimal("longitud").alias("longitud"),
        F.trim("sitio_de_e").alias("tipo_sitio"),
        F.trim("estatus").alias("estatus"),
        "_archivo_origen",
        "_fecha_ingesta",
    )
)

In [0]:
w = Window.partitionBy("num_cicloe").orderBy("id_registro")

In [0]:
clasificado = (
    tipado
    .withColumn("rn", F.row_number().over(w))
    .withColumn(
        "motivo_rechazo",
        F.when(F.col("num_cicloe").isNull(), "num_cicloe_invalido")
         .when(
             F.col("latitud").isNull() | F.col("longitud").isNull()
             | ~F.col("latitud").between(LAT_MIN, LAT_MAX)
             | ~F.col("longitud").between(LON_MIN, LON_MAX),
             "coordenadas_invalidas",
         )
         .when(F.col("rn") > 1, "duplicado")
    )
)

In [0]:
validas    = clasificado.filter(F.col("motivo_rechazo").isNull()).drop("rn", "motivo_rechazo")
cuarentena = clasificado.filter(F.col("motivo_rechazo").isNotNull()).drop("rn")

In [0]:
(
    validas.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_silver)
)

In [0]:
(
    cuarentena.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_cuarentena)
)

In [0]:
n_bronze     = spark.table(tabla_bronze).count()
n_rangos     = rangos.count()
n_silver     = spark.table(tabla_silver).count()
n_cuarentena = spark.table(tabla_cuarentena).count()

In [0]:
print(f"Filas en Bronce:              {n_bronze}")
print(f"Filas esperadas tras separar: {n_bronze + n_rangos}  (Bronce + un registro extra por cada rango)")
print(f"Filas en Silver:              {n_silver}")
print(f"Filas en cuarentena:          {n_cuarentena}")

Filas en Bronce:              677
Filas esperadas tras separar: 689  (Bronce + un registro extra por cada rango)
Filas en Silver:              689
Filas en cuarentena:          0


In [0]:
assert n_silver + n_cuarentena == n_bronze + n_rangos, "Se perdieron o duplicaron filas"

In [0]:
repetidas = (
    spark.table(tabla_silver).groupBy("num_cicloe").count().filter("count > 1").count()
)
assert repetidas == 0, f"Hay {repetidas} números de estación repetidos en Silver"

In [0]:
display(
    spark.table(tabla_cuarentena)
    .groupBy("motivo_rechazo").count().orderBy(F.desc("count"))
)

motivo_rechazo,count


In [0]:
display(spark.table(tabla_silver).orderBy("num_cicloe").limit(20))

id_registro,num_cicloe,num_cicloe_original,id_ubicacion,sistema,calle_principal,calle_secundaria,colonia,alcaldia,latitud,longitud,tipo_sitio,estatus,_archivo_origen,_fecha_ingesta
140,1,001,1,Ecobici,Rio Sena,Rio Balsas,Cuauhtemoc,Cuauhtemoc,19.4335895,-99.1678193,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
141,2,002,2,Ecobici,Rio Guadalquivir,Rio Nazas,Cuauhtemoc,Cuauhtemoc,19.43062276,-99.17123066,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
643,3,003,3,Ecobici,Av. Paseo De La Reforma,Av. Insurgentes,Tabacalera,Cuauhtemoc,19.43163,-99.158547,Banqueta,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
142,4,004,4,Ecobici,Rio Nilo,Rio Panuco,Cuauhtemoc,Cuauhtemoc,19.428491,-99.171693,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
143,5,005,5,Ecobici,Rio Panuco,Rio Tiber,Cuauhtemoc,Cuauhtemoc,19.429804,-99.169451,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
144,6,006,6,Ecobici,Rio Panuco,Rio Sena,Cuauhtemoc,Cuauhtemoc,19.430963,-99.166986,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
145,7,007,7,Ecobici,Rio Elba,Rio Lerma,Cuauhtemoc,Cuauhtemoc,19.425355,-99.17515,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
644,8,008,8,Ecobici,Av. Paseo De La Reforma,General Prim,Tabacalera,Cuauhtemoc,19.431256,-99.158153,Banqueta,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
146,9,009,9,Ecobici,Rio Guadalquivir,Rio Lerma,Cuauhtemoc,Cuauhtemoc,19.428184,-99.170013,Arroyo,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
645,10,010,10,Ecobici,Av. Paseo De La Reforma,Ramirez,Tabacalera,Cuauhtemoc,19.433431,-99.155142,Banqueta,Instalada,caracteristicas_cicloestaciones.json,2026-11-12T21:03:47.842569Z
